In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from pysr import PySRRegressor, TemplateExpressionSpec
import matplotlib.pyplot as plt


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
df = pd.read_excel("./banco de dados daniel.xlsx", "Dataset com TS variável")
df = df[df["w (mm)"] <= 0.4]
df = df.drop("Autor", axis=1)
X = df.drop("w (mm)", axis=1)
y = df["w (mm)"]

In [3]:
X.columns = ["d_linha", "p", "n", "c", "phi", "sigma", "e_phi", "f_t"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

In [5]:
template_configs = {
    "mul_combine": {
        "combine": "(f(d_linha)^a[1]) * (g(p)^a[2]) * (h(n)^a[3]) * (m(c)^a[4]) * (z(phi)^a[5]) * (y(sigma)^a[6]) * (t(e_phi)^a[7]) * (l(f_t)^a[8])"
    }
}

variable_names = ["d_linha", "p", "n", "c", "phi", "sigma", "e_phi", "f_t"]

base_config = {
    "binary_operators": ["+", "-", "*", "/"],
    "populations": 100,
    "population_size": 200,
    "ncycles_per_iteration": 500,
    "maxsize": 15,
    "constraints":{"pow": (0, 3)}
}

results = {}
for name, template_config in template_configs.items():
    if name in ["sum_equation", "mul_equation"]:
        expressions = ["f"]
    else:
        expressions = ["f", "g", "h", "m", "z", "y", "t", "l"]
    
    template = TemplateExpressionSpec(
        expressions=expressions,
        variable_names=variable_names,
        parameters={"a": len(variable_names)},
        combine=template_config["combine"]
    )
    
    model = PySRRegressor(expression_spec=template, **base_config)
    model.fit(X_train, y_train)
    
    results[name] = model
    print(f"Completed training: {name}")

/home/andre/programing/lamfo/ic_andre_rimes/env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
[ Info: Started!



Expressions evaluated per second: 0.000e+00
Progress: 0 / 10000 total iterations (0.000%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 8.530e+00
Progress: 1 / 10000 total iterations (0.010%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
─────────────────────────────────────────────────────────────────────────────

[ Info: Final population:
[ Info: Results saved to:


Completed training: mul_combine
  - outputs/20251013_120724_vQuqdq/hall_of_fame.csv


In [6]:
evaluation_results = {}

for model_name, model in results.items():
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)
    
    best_eq = model.get_best()
    best_equation = best_eq['equation']
    
    evaluation_results[model_name] = {
        'rmse_train': rmse_train,
        'rmse_test': rmse_test,
        'r2_train': r2_train,
        'r2_test': r2_test,
        'best_equation': best_equation,
    }

pd.set_option('display.max_colwidth', None)
results_df = pd.DataFrame(evaluation_results).T
results_df

,rmse_train,rmse_test,r2_train,r2_test,best_equation
mul_combine,0.050172,0.045738,0.416556,0.127165,"f = #1; g = #1; h = 0.22166209; m = #1; z = #1; y = #1; t = #1; l = #1 + 8.353253; a = [0.6426112, 0.08412056, 5.1286497, 0.24424586, -0.08111269, 0.29831624, -0.2827823, 0.42952615]"


In [7]:
from IPython.display import display, Latex

def display_latex(equation_latex,r2_train, r2_test, rmse_train,rmse_test):
    table_latex = f"""
        \\begin{{array}}{{|l|c|}}
        \\hline
        \\textbf{{Metric}} & \\textbf{{Value}} \\\\
        \\hline
        \\text{{Equation}} & {equation_latex} \\\\
        \\hline
        R^2 \\text{{ (Train)}} & {r2_train:.4f} \\\\
        \\hline
        R^2 \\text{{ (Test)}} & {r2_test:.4f} \\\\
        \\hline
        \\text{{RMSE (Train)}} & {rmse_train:.4f} \\\\
        \\hline
        \\text{{RMSE (Test)}} & {rmse_test:.4f} \\\\
        \\hline
        \\end{{array}}
        """
    
    display(Latex(table_latex))

args_map = {
    'f': 'd_linha',
    'g': 'p',
    'h': 'n',
    'm': 'c',
    'z': 'phi',
    'y': 'sigma',
    't': 'e_phi',
    'l': 'f_t'
}

In [8]:
import ast

def parse_list_equation(best_equation, variable_names, separator):
    """Parse equations with list format and create LaTeX."""
    
    list_string = best_equation[1].strip()
    list_part = list_string.split(" = ", 1)[1]
    actual_list = ast.literal_eval(list_part)
    
    terms = [f"{var}^{{{actual_list[idx]}}}" for idx, var in enumerate(variable_names)]
    return separator.join(terms)

def parse_combine_equation(best_equation, args_map):
    """Parse combine equations with function composition and create LaTeX."""
    
    a = ast.literal_eval(
        next(expr.strip().split("=")[-1].strip() 
             for expr in best_equation 
             if expr.strip().split("=")[0].strip() == 'a')
    )
    
    functions_to_process = [expr for expr in best_equation 
                           if expr.strip().split("=")[0].strip() != 'a']

    elements = []
    for idx, expression in enumerate(functions_to_process):
        equation = "("
        expression = expression.strip().split("=")
        function_name = expression[0].strip()
        function_expression = expression[-1].strip().split()

        for exp in function_expression:
            equation += args_map[function_name] if exp == "#1" else exp
                
        equation += f")^{{{a[idx]}}}"  
        elements.append(equation)

    return elements

equation_configs = {
    'sum_equation': {
        'parser': lambda eq: parse_list_equation(eq, variable_names, " \\cdot "),
        'display': False
    },
    'mul_equation': {
        'parser': lambda eq: parse_list_equation(eq, variable_names, " + "),
        'display': False
    },
    'mul_combine': {
        'parser': lambda eq: " \\cdot ".join(parse_combine_equation(eq, args_map)),
        'display': True
    },
    'sum_combine': {
        'parser': lambda eq: " + ".join(parse_combine_equation(eq, args_map)),
        'display': False
    }
}

for name, row in results_df.iterrows():
    best_equation = row['best_equation'].split(";")
    
    if name in equation_configs:
        config = equation_configs[name]
        latex_equation = config['parser'](best_equation)
        
        if config['display']:
            display_latex(
                equation_latex=latex_equation,
                r2_train=row['r2_train'],
                r2_test=row['r2_test'],
                rmse_train=row['rmse_train'],
                rmse_test=row['rmse_test'],
            )


<IPython.core.display.Latex object>